### Environment Setup

In [19]:
import os
import sys
import subprocess
import platform
import torch
import triton
import triton.language as tl
import triton.testing
import numpy as np

print("=" * 65)
print("Module 2: IR Optimization Pass Implementation")
print("=" * 65)

print(f"Python:         {sys.version.split()[0]}")
print(f"Platform:       {platform.system()}")
print(f"PyTorch:        {torch.__version__}")
print(f"Triton:         {triton.__version__}")
print(f"CUDA runtime:   {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

assert torch.cuda.is_available(), "Please enable a GPU in Colab."

gpu = torch.cuda.get_device_properties(0)

print(f"GPU:            {gpu.name}")
print(f"Compute Cap.:   {gpu.major}.{gpu.minor}")
print(f"GPU Memory:     {gpu.total_memory / 1024**3:.2f} GB")

# Check C++ compiler
result = subprocess.run(
    ["g++", "--version"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"C++ Compiler:   {result.stdout.splitlines()[0]}")
else:
    print("C++ Compiler:   Not found")

print("=" * 65)

Module 2: IR Optimization Pass Implementation
Python:         3.13.16
Platform:       Linux
PyTorch:        2.11.0+cu130
Triton:         3.6.0
CUDA runtime:   13.0
CUDA available: True
GPU:            Tesla T4
Compute Cap.:   7.5
GPU Memory:     14.56 GB
C++ Compiler:   g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0


### Triton GEMM Kernel

In [20]:
@triton.jit
def gemm_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):

    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # Row and column offsets
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    # FP32 accumulator
    acc = tl.full(
        (BLOCK_M, BLOCK_N),
        0,
        dtype=tl.float32
    )

    # Loop over K dimension
    for k in range(tl.cdiv(K, BLOCK_K)):

        k_offsets = k * BLOCK_K + offs_k

        a_ptrs = (
            A_ptr
            + offs_m[:, None] * K
            + k_offsets[None, :]
        )

        b_ptrs = (
            B_ptr
            + k_offsets[:, None] * N
            + offs_n[None, :]
        )

        a = tl.load(
            a_ptrs,
            mask=(
                (offs_m[:, None] < M)
                & (k_offsets[None, :] < K)
            ),
            other=0
        )

        b = tl.load(
            b_ptrs,
            mask=(
                (k_offsets[:, None] < K)
                & (offs_n[None, :] < N)
            ),
            other=0
        )

        acc = tl.dot(a, b, acc)

    # Convert FP32 accumulator to FP16
    result = acc.to(tl.float16)

    c_ptrs = (
        C_ptr
        + offs_m[:, None] * N
        + offs_n[None, :]
    )

    tl.store(
        c_ptrs,
        result,
        mask=(
            (offs_m[:, None] < M)
            & (offs_n[None, :] < N)
        )
    )

### GEMM Wrapper

In [21]:
def triton_gemm(
    A,
    B,
    block_m=32,
    block_n=64,
    block_k=32,
    return_kernel=False
):

    M, K = A.shape
    K2, N = B.shape

    assert K == K2
    assert A.is_cuda and B.is_cuda
    assert A.dtype == torch.float16
    assert B.dtype == torch.float16
    assert A.is_contiguous() and B.is_contiguous()

    C = torch.empty(
        (M, N),
        device=A.device,
        dtype=torch.float16
    )

    grid = (
        triton.cdiv(M, block_m),
        triton.cdiv(N, block_n)
    )

    kernel = gemm_kernel[grid](
        A,
        B,
        C,
        M,
        N,
        K,
        BLOCK_M=block_m,
        BLOCK_N=block_n,
        BLOCK_K=block_k,
        num_warps=4
    )

    if return_kernel:
        return C, kernel

    return C

### Correctness Verification

In [22]:
torch.manual_seed(42)

M = 512
N = 512
K = 512

A = torch.randn(
    (M, K),
    device="cuda",
    dtype=torch.float16
)

B = torch.randn(
    (K, N),
    device="cuda",
    dtype=torch.float16
)

# PyTorch reference
C_ref = torch.matmul(A, B)

# Triton GEMM
C_triton, compiled_kernel = triton_gemm(
    A,
    B,
    return_kernel=True
)

torch.cuda.synchronize()

max_error = (
    C_ref.float() - C_triton.float()
).abs().max().item()

print("=" * 60)
print("GEMM Correctness Verification")
print("=" * 60)

print(f"Matrix shape: {M} x {N} x {K}")
print(f"Max abs error: {max_error:.6f}")

torch.testing.assert_close(
    C_triton,
    C_ref,
    atol=0.125,
    rtol=0.01
)

print("Correctness: PASSED")

print(f"Compiled kernel: {compiled_kernel.n_regs} registers/thread")
print(f"Shared memory:   {compiled_kernel.metadata.shared} bytes")

GEMM Correctness Verification
Matrix shape: 512 x 512 x 512
Max abs error: 0.031250
Correctness: PASSED
Compiled kernel: 189 registers/thread
Shared memory:   8192 bytes


In [23]:
print("=" * 65)
print("Triton Intermediate Representation (TTIR)")
print("=" * 65)

print("Available IR stages:")
print(list(compiled_kernel.asm.keys()))

ttir = compiled_kernel.asm["ttir"]

print("\nFirst 100 lines of TTIR:")
print("\n".join(ttir.splitlines()[:100]))

Triton Intermediate Representation (TTIR)
Available IR stages:
['source', 'ttir', 'ttgir', 'llir', 'ptx', 'cubin']

First 100 lines of TTIR:
#loc = loc("/tmp/ipykernel_512/4156870778.py":2:0)
#loc45 = loc("A_ptr"(#loc))
#loc46 = loc("B_ptr"(#loc))
#loc47 = loc("C_ptr"(#loc))
module {
  tt.func public @gemm_kernel(%A_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("A_ptr"(#loc)), %B_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("B_ptr"(#loc)), %C_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("C_ptr"(#loc))) attributes {noinline = false} {
    %c16_i32 = arith.constant 16 : i32 loc(#loc1)
    %cst = arith.constant dense<0.000000e+00> : tensor<32x64xf16> loc(#loc1)
    %cst_0 = arith.constant dense<0.000000e+00> : tensor<32x32xf16> loc(#loc1)
    %c1_i32 = arith.constant 1 : i32 loc(#loc2)
    %c0_i32 = arith.constant 0 : i32 loc(#loc2)
    %cst_1 = arith.constant dense<512> : tensor<1x64xi32> loc(#loc1)
    %cst_2 = arith.constant dense<512> : tensor<1x32xi32> loc(#loc1)
   

In [24]:
# Cell 9 — TTIR Operation Analyzer

from collections import Counter
import re

ttir = compiled_kernel.asm["ttir"]

operations = []

# Match common MLIR operation names
op_pattern = re.compile(
    r"\b((?:arith|tt|scf|math)\.[a-zA-Z0-9_]+)\b"
)

for line in ttir.splitlines():

    line = line.strip()

    if not line or line.startswith("#"):
        continue

    matches = op_pattern.findall(line)

    if matches:
        operations.append(matches[0])

counts = Counter(operations)

print("=" * 65)
print("GEMM TTIR Operation Statistics")
print("=" * 65)

for op, count in counts.most_common():
    print(f"{op:<25} {count:>5}")

print("-" * 65)
print(f"Total matched operations: {len(operations)}")

GEMM TTIR Operation Statistics
tt.broadcast                 12
arith.constant               11
arith.muli                    6
tt.splat                      6
tt.expand_dims                6
tt.addptr                     6
arith.cmpi                    6
arith.addi                    3
arith.andi                    3
tt.get_program_id             2
tt.make_range                 2
tt.load                       2
tt.func                       1
scf.for                       1
tt.dot                        1
scf.yield                     1
arith.truncf                  1
tt.store                      1
tt.return                     1
-----------------------------------------------------------------
Total matched operations: 72


In [25]:
# Cell 10 — Inspect Loop Body

lines = ttir.splitlines()

loop_start = next(
    i for i, line in enumerate(lines)
    if "scf.for" in line
)

depth = 0
loop_end = None

for i in range(loop_start, len(lines)):
    line = lines[i]
    depth += line.count("{") - line.count("}")

    if depth == 0:
        loop_end = i
        break

assert loop_end is not None, "Loop end not found"

loop_lines = lines[loop_start:loop_end + 1]

print("=" * 65)
print("GEMM Loop Body")
print("=" * 65)

for line in loop_lines:
    print(line)

GEMM Loop Body
    %acc_10 = scf.for %k = %c0_i32 to %c16_i32 step %c1_i32 iter_args(%acc_18 = %acc) -> (tensor<32x64xf32>)  : i32 {
      %k_offsets = arith.muli %k, %c32_i32 : i32 loc(#loc58)
      %k_offsets_19 = tt.splat %k_offsets : i32 -> tensor<32xi32> loc(#loc59)
      %k_offsets_20 = arith.addi %k_offsets_19, %offs_m_4 : tensor<32xi32> loc(#loc59)
      %a_ptrs = tt.expand_dims %offs_m_6 {axis = 1 : i32} : tensor<32xi32> -> tensor<32x1xi32> loc(#loc60)
      %a_ptrs_21 = arith.muli %a_ptrs, %cst_3 : tensor<32x1xi32> loc(#loc61)
      %a_ptrs_22 = tt.splat %A_ptr : !tt.ptr<f16> -> tensor<32x1x!tt.ptr<f16>> loc(#loc62)
      %a_ptrs_23 = tt.addptr %a_ptrs_22, %a_ptrs_21 : tensor<32x1x!tt.ptr<f16>>, tensor<32x1xi32> loc(#loc62)
      %a_ptrs_24 = tt.expand_dims %k_offsets_20 {axis = 0 : i32} : tensor<32xi32> -> tensor<1x32xi32> loc(#loc63)
      %a_ptrs_25 = tt.broadcast %a_ptrs_23 : tensor<32x1x!tt.ptr<f16>> -> tensor<32x32x!tt.ptr<f16>> loc(#loc64)
      %a_ptrs_26 = tt.broadca

In [26]:
# Cell 11 — Loop-Invariant Dependency Analysis

import re

# Our loop body comes from Cell 10
body = loop_lines[1:-1]

# Loop induction variable and loop-carried accumulator
variant_values = {"%k", "%acc_18"}

# All definitions local to the loop
definitions = {}

for line in body:
    line = line.strip()

    match = re.match(
        r"^(%[\w.]+)\s*=\s*(.*)",
        line
    )

    if match:
        result = match.group(1)
        expression = match.group(2)
        definitions[result] = expression

# Find SSA operands from an expression
def get_operands(expression):

    # Remove location metadata and result type annotations
    expression = expression.split(" loc(")[0]
    expression = expression.split(" : ")[0]

    return re.findall(r"%[\w.]+", expression)


invariant = {}
variant = {}

for result, expression in definitions.items():

    operands = get_operands(expression)

    is_variant = any(
        operand in variant_values
        for operand in operands
    )

    if is_variant:
        variant[result] = expression
        variant_values.add(result)
    else:
        invariant[result] = expression

print("=" * 65)
print("Loop-Invariant Analysis")
print("=" * 65)

print("\nPotential loop-invariant operations:\n")

for result, expression in invariant.items():
    print(f"{result} = {expression}")

print("\n" + "-" * 65)

print(f"Loop-invariant candidates: {len(invariant)}")
print(f"Loop-dependent operations: {len(variant)}")

Loop-Invariant Analysis

Potential loop-invariant operations:

%a_ptrs = tt.expand_dims %offs_m_6 {axis = 1 : i32} : tensor<32xi32> -> tensor<32x1xi32> loc(#loc60)
%a_ptrs_21 = arith.muli %a_ptrs, %cst_3 : tensor<32x1xi32> loc(#loc61)
%a_ptrs_22 = tt.splat %A_ptr : !tt.ptr<f16> -> tensor<32x1x!tt.ptr<f16>> loc(#loc62)
%a_ptrs_23 = tt.addptr %a_ptrs_22, %a_ptrs_21 : tensor<32x1x!tt.ptr<f16>>, tensor<32x1xi32> loc(#loc62)
%a_ptrs_25 = tt.broadcast %a_ptrs_23 : tensor<32x1x!tt.ptr<f16>> -> tensor<32x32x!tt.ptr<f16>> loc(#loc64)
%b_ptrs_29 = tt.splat %B_ptr : !tt.ptr<f16> -> tensor<32x1x!tt.ptr<f16>> loc(#loc67)
%b_ptrs_31 = tt.expand_dims %offs_n_9 {axis = 0 : i32} : tensor<64xi32> -> tensor<1x64xi32> loc(#loc68)
%b_ptrs_33 = tt.broadcast %b_ptrs_31 : tensor<1x64xi32> -> tensor<32x64xi32> loc(#loc69)
%a = arith.cmpi slt, %a_ptrs, %cst_3 : tensor<32x1xi32> loc(#loc70)
%a_36 = tt.broadcast %a : tensor<32x1xi1> -> tensor<32x32xi1> loc(#loc72)
%b_40 = arith.cmpi slt, %b_ptrs_31, %cst_1 : tens

In [27]:
# Cell 12 — Def-Use Graph + LICM Candidate Classification

import re
from collections import defaultdict

instructions = {}

for line in loop_lines[1:-1]:
    line = line.strip()

    match = re.match(r"^(%[\w.]+)\s*=\s*(.*)", line)
    if not match:
        continue

    name, rhs = match.groups()

    # Remove debug location and result-type information
    expression = rhs.split(" loc(")[0]
    operation_part = expression.split(" : ")[0]

    op_match = re.match(
        r"^([a-zA-Z_][\w.]+)", operation_part
    )
    if not op_match:
        continue

    op = op_match.group(1)
    operands = re.findall(r"%[\w.]+", operation_part)

    instructions[name] = {
        "op": op,
        "operands": operands,
        "line": line
    }

# Build users mapping
users = defaultdict(list)

for name, inst in instructions.items():
    for operand in inst["operands"]:
        users[operand].append(name)

# Known values that change across loop iterations
variant_values = {"%k", "%acc_18"}

# Conservative whitelist for this experiment.
# These operations still require proper MLIR legality checks
# before being physically moved.
pure_candidate_ops = {
    "arith.muli",
    "arith.cmpi",
    "tt.expand_dims",
    "tt.splat",
    "tt.broadcast",
}

classification = {}
hoist_candidates = []

for name, inst in instructions.items():

    operands = inst["operands"]

    is_invariant = all(
        operand not in variant_values
        for operand in operands
    )

    if not is_invariant:
        variant_values.add(name)
        classification[name] = "LOOP_DEPENDENT"
        continue

    if inst["op"] in pure_candidate_ops:
        classification[name] = "POTENTIAL_HOIST"
        hoist_candidates.append(name)

    else:
        classification[name] = "NEEDS_REVIEW"

print("=" * 75)
print("GEMM LICM Candidate Analysis")
print("=" * 75)

for name, inst in instructions.items():

    status = classification[name]

    if status == "LOOP_DEPENDENT":
        continue

    print(f"\nValue:  {name}")
    print(f"Op:     {inst['op']}")
    print(f"Inputs: {inst['operands']}")
    print(f"Users:  {users[name]}")
    print(f"Status: {status}")

print("\n" + "=" * 75)
print(f"Potential hoist candidates: {len(hoist_candidates)}")
print(f"Requires further review:    {list(classification.values()).count('NEEDS_REVIEW')}")

GEMM LICM Candidate Analysis

Value:  %a_ptrs
Op:     tt.expand_dims
Inputs: ['%offs_m_6']
Users:  ['%a_ptrs_21', '%a']
Status: POTENTIAL_HOIST

Value:  %a_ptrs_21
Op:     arith.muli
Inputs: ['%a_ptrs', '%cst_3']
Users:  ['%a_ptrs_23']
Status: POTENTIAL_HOIST

Value:  %a_ptrs_22
Op:     tt.splat
Inputs: ['%A_ptr']
Users:  ['%a_ptrs_23']
Status: POTENTIAL_HOIST

Value:  %a_ptrs_23
Op:     tt.addptr
Inputs: ['%a_ptrs_22', '%a_ptrs_21']
Users:  ['%a_ptrs_25']
Status: NEEDS_REVIEW

Value:  %a_ptrs_25
Op:     tt.broadcast
Inputs: ['%a_ptrs_23']
Users:  ['%a_ptrs_27']
Status: POTENTIAL_HOIST

Value:  %b_ptrs_29
Op:     tt.splat
Inputs: ['%B_ptr']
Users:  ['%b_ptrs_30']
Status: POTENTIAL_HOIST

Value:  %b_ptrs_31
Op:     tt.expand_dims
Inputs: ['%offs_n_9']
Users:  ['%b_ptrs_33', '%b_40']
Status: POTENTIAL_HOIST

Value:  %b_ptrs_33
Op:     tt.broadcast
Inputs: ['%b_ptrs_31']
Users:  ['%b_ptrs_34']
Status: POTENTIAL_HOIST

Value:  %a
Op:     arith.cmpi
Inputs: ['%a_ptrs', '%cst_3']
Users:  ['%

### Capture Compiler Pass Dumps

In [28]:
import inspect
import textwrap
from pathlib import Path

# Export our existing Triton GEMM implementation
kernel_source = textwrap.dedent(
    inspect.getsource(gemm_kernel.fn)
)

script = f"""
import torch
import triton
import triton.language as tl

{kernel_source}

M = N = K = 512

A = torch.randn(
    (M, K), device="cuda", dtype=torch.float16
)
B = torch.randn(
    (K, N), device="cuda", dtype=torch.float16
)
C = torch.empty(
    (M, N), device="cuda", dtype=torch.float16
)

grid = (triton.cdiv(M, 32), triton.cdiv(N, 64))

gemm_kernel[grid](
    A, B, C,
    M, N, K,
    BLOCK_M=32,
    BLOCK_N=64,
    BLOCK_K=32,
    num_warps=4
)

torch.cuda.synchronize()

torch.testing.assert_close(
    C,
    torch.matmul(A, B),
    atol=0.125,
    rtol=0.01
)

print("GEMM compilation and correctness: PASSED")
"""

Path("gemm_pass_debug.py").write_text(script)

print("Created gemm_pass_debug.py")

Created gemm_pass_debug.py


### Run Compiler with IR Dump

In [29]:
import os
import subprocess
from pathlib import Path

dump_path = Path("gemm_mlir_passes.log").resolve()

env = os.environ.copy()

env["MLIR_ENABLE_DUMP"] = "gemm_kernel"
env["TRITON_ALWAYS_COMPILE"] = "1"
env["MLIR_DUMP_PATH"] = str(dump_path)

result = subprocess.run(
    ["python", "gemm_pass_debug.py"],
    env=env,
    capture_output=True,
    text=True,
    timeout=180
)

print("Return code:", result.returncode)
print("Stdout:", result.stdout[-1000:])

if result.returncode != 0:
    print("Stderr (last 3000 chars):")
    print(result.stderr[-3000:])

if dump_path.exists():
    print("Dump file:", dump_path)
    print("Dump size:", dump_path.stat().st_size, "bytes")
else:
    print("No dump file generated.")

Return code: 0
Stdout: GEMM compilation and correctness: PASSED

Dump file: /content/gemm_mlir_passes.log
Dump size: 4086071 bytes


### Inspect Compiler Pass Names

In [30]:
import re
from pathlib import Path

dump_path = Path("gemm_mlir_passes.log")

if dump_path.exists():

    dump = dump_path.read_text(errors="replace")

    lines = dump.splitlines()

    headers = [
        line.strip()
        for line in lines
        if (
            "IR Dump" in line
            or "Before" in line and "Pass" in line
        )
    ]

    print("=" * 65)
    print("Compiler Pass Dump Headers")
    print("=" * 65)

    for line in headers[:60]:
        print(line[:180])

    print("\nTotal headers found:", len(headers))

    for keyword in [
        "licm",
        "cse",
        "canonical",
        "loop",
        "combine"
    ]:
        matched = [
            line for line in headers
            if keyword in line.lower()
        ]

        print(f"{keyword:<15}: {len(matched)}")

else:
    print("Please check Cell 14 output.")

Compiler Pass Dump Headers
// -----// IR Dump Before Inliner (inline) ('builtin.module' operation) //----- //
// -----// IR Dump Before Canonicalizer (canonicalize) ('tt.func' operation: @gemm_kernel) //----- //
// -----// IR Dump Before Canonicalizer (canonicalize) ('tt.func' operation: @gemm_kernel) //----- //
// -----// IR Dump Before TritonRewriteTensorPointer (triton-rewrite-tensor-pointer) ('builtin.module' operation) //----- //
// -----// IR Dump Before TritonRewriteTensorDescriptorToPointer (triton-rewrite-tensor-descriptor-to-pointer) ('builtin.module' operation) //----- //
// -----// IR Dump Before Canonicalizer (canonicalize) ('builtin.module' operation) //----- //
// -----// IR Dump Before TritonCombineOps (triton-combine) ('builtin.module' operation) //----- //
// -----// IR Dump Before TritonReorderBroadcast (triton-reorder-broadcast) ('builtin.module' operation) //----- //
// -----// IR Dump Before CSE (cse) ('builtin.module' operation) //----- //
// -----// IR Dump Befo

### Extract IR Before and After LICM

In [31]:
# Cell 16 — Extract LICM Before / After IR

import re
from pathlib import Path

dump = Path("gemm_mlir_passes.log").read_text(
    errors="replace"
)

header_pattern = re.compile(
    r"(?m)^// -+// IR Dump Before .+ //-+ //\s*$"
)

matches = list(header_pattern.finditer(dump))

snapshots = []

for i, match in enumerate(matches):
    start = match.end()

    end = (
        matches[i + 1].start()
        if i + 1 < len(matches)
        else len(dump)
    )

    snapshots.append({
        "header": match.group().strip(),
        "ir": dump[start:end].strip()
    })

licm_indices = [
    i for i, snapshot in enumerate(snapshots)
    if "TritonLoopInvariantCodeMotion" in snapshot["header"]
]

assert len(licm_indices) == 1, (
    f"Expected one LICM snapshot, found {len(licm_indices)}"
)

idx = licm_indices[0]

assert idx + 1 < len(snapshots)

before_licm = snapshots[idx]["ir"]
after_licm = snapshots[idx + 1]["ir"]

Path("before_licm.mlir").write_text(before_licm)
Path("after_licm.mlir").write_text(after_licm)

print("Before:", snapshots[idx]["header"])
print("After: ", snapshots[idx + 1]["header"])

print("\nBefore size:", len(before_licm), "characters")
print("After size: ", len(after_licm), "characters")

print("\nFiles saved:")
print("before_licm.mlir")
print("after_licm.mlir")

Before: // -----// IR Dump Before TritonLoopInvariantCodeMotion (triton-licm) ('builtin.module' operation) //----- //
After:  // -----// IR Dump Before Canonicalizer (canonicalize) ('builtin.module' operation) //----- //

Before size: 12985 characters
After size:  12964 characters

Files saved:
before_licm.mlir
after_licm.mlir


### Compare LICM Transformations

In [32]:
# Cell 17 — LICM IR Diff

import difflib

before_lines = before_licm.splitlines()
after_lines = after_licm.splitlines()

diff = list(
    difflib.unified_diff(
        before_lines,
        after_lines,
        fromfile="Before LICM",
        tofile="After LICM",
        lineterm=""
    )
)

print("=" * 70)
print("LICM Transformation Analysis")
print("=" * 70)

print("Before lines:", len(before_lines))
print("After lines: ", len(after_lines))
print("Diff lines:  ", len(diff))

if not diff:
    print("\nNo textual changes detected.")
    print("LICM may have made no changes for this GEMM.")
else:
    print("\nFirst 100 diff lines:\n")
    print("\n".join(diff[:100]))

LICM Transformation Analysis
Before lines: 176
After lines:  176
Diff lines:   203

First 100 diff lines:

--- Before LICM
+++ After LICM
@@ -33,63 +33,63 @@
     %offs_n_12 = tt.make_range {end = 64 : i32, start = 0 : i32} : tensor<64xi32, #ttg.slice<{dim = 0, parent = #blocked1}>> loc(#loc55)
     %offs_n_13 = tt.splat %offs_n : i32 -> tensor<64xi32, #ttg.slice<{dim = 0, parent = #blocked1}>> loc(#loc56)
     %offs_n_14 = arith.addi %offs_n_13, %offs_n_12 : tensor<64xi32, #ttg.slice<{dim = 0, parent = #blocked1}>> loc(#loc56)
-    %acc_15 = scf.for %acc_23 = %c0_i32 to %c16_i32 step %c1_i32 iter_args(%acc_24 = %acc) -> (tensor<32x64xf32, #blocked>)  : i32 {
-      %k_offsets = arith.muli %acc_23, %c32_i32 : i32 loc(#loc58)
-      %k_offsets_25 = tt.splat %k_offsets : i32 -> tensor<32xi32, #ttg.slice<{dim = 0, parent = #blocked2}>> loc(#loc59)
-      %k_offsets_26 = tt.splat %k_offsets : i32 -> tensor<32xi32, #ttg.slice<{dim = 1, parent = #blocked1}>> loc(#loc59)
-      %k_offsets_27 

### 2.4 GEMM CSE Experiment

In [33]:
# Cell 18 — Find CSE Passes

cse_indices = [
    i for i, snapshot in enumerate(snapshots)
    if (
        "Before CSE (cse)" in snapshot["header"]
        or "Before TritonLoopAwareCSE" in snapshot["header"]
    )
]

print("=" * 70)
print("CSE Passes in Triton GEMM Compiler Pipeline")
print("=" * 70)

for n, idx in enumerate(cse_indices):
    print(f"\nCSE Pass #{n}")
    print("Before:", snapshots[idx]["header"])

    if idx + 1 < len(snapshots):
        print("After: ", snapshots[idx + 1]["header"])

print("\nTotal CSE passes:", len(cse_indices))

CSE Passes in Triton GEMM Compiler Pipeline

CSE Pass #0
Before: // -----// IR Dump Before CSE (cse) ('builtin.module' operation) //----- //
After:  // -----// IR Dump Before SymbolDCE (symbol-dce) ('builtin.module' operation) //----- //

CSE Pass #1
Before: // -----// IR Dump Before TritonLoopAwareCSE (triton-loop-aware-cse) ('builtin.module' operation) //----- //
After:  // -----// IR Dump Before TritonLoopInvariantCodeMotion (triton-licm) ('builtin.module' operation) //----- //

CSE Pass #2
Before: // -----// IR Dump Before TritonLoopAwareCSE (triton-loop-aware-cse) ('builtin.module' operation) //----- //
After:  // -----// IR Dump Before TritonGPUPrefetch (tritongpu-prefetch) ('builtin.module' operation) //----- //

CSE Pass #3
Before: // -----// IR Dump Before TritonLoopAwareCSE (triton-loop-aware-cse) ('builtin.module' operation) //----- //
After:  // -----// IR Dump Before SymbolDCE (symbol-dce) ('builtin.module' operation) //----- //

CSE Pass #4
Before: // -----// IR Dump Befo

In [34]:
# Cell 19 — Compare First CSE Pass
# Compare CSE Before and After

import difflib
from pathlib import Path

first_cse_idx = next(
    i for i in cse_indices
    if "Before CSE (cse)" in snapshots[i]["header"]
)

before_cse = snapshots[first_cse_idx]["ir"]
after_cse = snapshots[first_cse_idx + 1]["ir"]

Path("before_cse.mlir").write_text(before_cse)
Path("after_cse.mlir").write_text(after_cse)

diff = list(
    difflib.unified_diff(
        before_cse.splitlines(),
        after_cse.splitlines(),
        fromfile="Before CSE",
        tofile="After CSE",
        lineterm=""
    )
)

print("=" * 70)
print("GEMM CSE Transformation Analysis")
print("=" * 70)

print("Before IR lines:", len(before_cse.splitlines()))
print("After IR lines: ", len(after_cse.splitlines()))
print("Diff lines:      ", len(diff))

if diff:
    print("\n".join(diff[:120]))
else:
    print("\nNo textual IR changes detected.")

GEMM CSE Transformation Analysis
Before IR lines: 180
After IR lines:  161
Diff lines:       308
--- Before CSE
+++ After CSE
@@ -1,7 +1,7 @@
 #loc = loc("/content/gemm_pass_debug.py":7:0)
-#loc52 = loc("A_ptr"(#loc))
-#loc53 = loc("B_ptr"(#loc))
-#loc54 = loc("C_ptr"(#loc))
+#loc45 = loc("A_ptr"(#loc))
+#loc46 = loc("B_ptr"(#loc))
+#loc47 = loc("C_ptr"(#loc))
 module {
   tt.func public @gemm_kernel(%A_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("A_ptr"(#loc)), %B_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("B_ptr"(#loc)), %C_ptr: !tt.ptr<f16> {tt.divisibility = 16 : i32} loc("C_ptr"(#loc))) attributes {noinline = false} {
     %c16_i32 = arith.constant 16 : i32 loc(#loc1)
@@ -12,77 +12,70 @@
     %cst_1 = arith.constant dense<512> : tensor<1x64xi32> loc(#loc1)
     %cst_2 = arith.constant dense<512> : tensor<1x32xi32> loc(#loc1)
     %cst_3 = arith.constant dense<512> : tensor<32x1xi32> loc(#loc1)
-    %acc = arith.constant dense<0.000000e+00> : tensor<32x64xf32> loc(#l

In [35]:
# Cell 20 — CSE Operation Statistics

import re
from collections import Counter

pattern = re.compile(
    r"^\s*(?:%[\w.]+\s*=\s*)?"
    r"((?:arith|tt|ttg|scf)\.[\w.]+)\b",
    re.MULTILINE
)

def count_operations(ir):
    return Counter(pattern.findall(ir))

before_counts = count_operations(before_cse)
after_counts = count_operations(after_cse)

print("=" * 70)
print("GEMM CSE Operation Count Comparison")
print("=" * 70)

print(
    f"{'Operation':<30}"
    f"{'Before':>10}"
    f"{'After':>10}"
    f"{'Delta':>10}"
)

for op in sorted(set(before_counts) | set(after_counts)):

    before = before_counts[op]
    after = after_counts[op]

    if before != after:
        print(
            f"{op:<30}"
            f"{before:>10}"
            f"{after:>10}"
            f"{after-before:>10}"
        )

print("-" * 70)
print("Total before:", sum(before_counts.values()))
print("Total after: ", sum(after_counts.values()))

GEMM CSE Operation Count Comparison
Operation                         Before     After     Delta
tt.expand_dims                        12         6        -6
tt.make_range                          3         2        -1
----------------------------------------------------------------------
Total before: 79
Total after:  72


### Implement GEMM TTIR CSE Prototype

In [36]:
# Cell 21 — Scope-Aware GEMM TTIR CSE Prototype

import re
from pathlib import Path
from collections import Counter

original_ir = Path("before_cse.mlir").read_text()

# First version: only explicitly supported, pure operations
SUPPORTED_OPS = {
    "tt.expand_dims",
    "tt.make_range",
}

definition_pattern = re.compile(
    r"^(\s*)(%[\w.]+)\s*=\s*(.*)$"
)

ssa_pattern = re.compile(r"%[\w.]+")


def replace_ssa(text, aliases):
    """Replace SSA uses with their canonical values."""
    def resolve(name):
        visited = set()

        while name in aliases and name not in visited:
            visited.add(name)
            name = aliases[name]

        return name

    return ssa_pattern.sub(
        lambda m: resolve(m.group(0)),
        text
    )


def run_shape_cse(ir):

    output = []
    aliases = {}

    # One expression table per lexical region
    expression_tables = [{}]

    removed = []

    for line in ir.splitlines():

        stripped = line.strip()

        # Enter SCF loop region
        if re.search(r"\bscf\.for\b.*\{\s*$", stripped):
            output.append(line)
            expression_tables.append({})
            continue

        # Exit SCF loop region
        if stripped.startswith("} loc(") and len(expression_tables) > 1:
            expression_tables.pop()
            output.append(line)
            continue

        match = definition_pattern.match(line)

        if not match:
            # Rewrite SSA uses in non-definition instructions,
            # such as tt.store or scf.yield.
            if not stripped.startswith("#"):
                line = replace_ssa(line, aliases)
            output.append(line)
            continue

        indent, result, rhs = match.groups()

        # Exclude debug locations from equivalence checking
        core = rhs.split(" loc(")[0]

        op_match = re.match(
            r"^([\w.]+)\b", core
        )

        if not op_match:
            output.append(line)
            continue

        op = op_match.group(1)

        # Rewrite operands, without rewriting the result name
        rhs = replace_ssa(rhs, aliases)
        core = rhs.split(" loc(")[0]

        if op not in SUPPORTED_OPS:
            output.append(f"{indent}{result} = {rhs}")
            continue

        # Full expression includes operation, operands,
        # attributes and result types.
        key = core

        current_table = expression_tables[-1]

        if key in current_table:

            canonical = current_table[key]
            aliases[result] = canonical

            removed.append({
                "operation": op,
                "duplicate": result,
                "canonical": canonical
            })

            continue

        current_table[key] = result
        output.append(f"{indent}{result} = {rhs}")

    return "\n".join(output) + "\n", removed


optimized_ir, removed = run_shape_cse(original_ir)

Path("prototype_cse.mlir").write_text(optimized_ir)

print("=" * 65)
print("GEMM TTIR CSE Prototype")
print("=" * 65)

for item in removed:
    print(
        f"{item['operation']:<20} "
        f"{item['duplicate']} -> {item['canonical']}"
    )

print("\nRemoved:", len(removed))
print("Breakdown:", dict(Counter(x["operation"] for x in removed)))

GEMM TTIR CSE Prototype
tt.make_range        %offs_k -> %offs_m_4
tt.expand_dims       %a -> %a_ptrs
tt.expand_dims       %a_36 -> %a_ptrs_24
tt.expand_dims       %b -> %b_ptrs
tt.expand_dims       %b_43 -> %b_ptrs_31
tt.expand_dims       %0 -> %c_ptrs
tt.expand_dims       %2 -> %c_ptrs_14

Removed: 7
Breakdown: {'tt.make_range': 1, 'tt.expand_dims': 6}


In [37]:
# Cell 22 — Prototype CSE Verification

import difflib

diff = list(
    difflib.unified_diff(
        original_ir.splitlines(),
        optimized_ir.splitlines(),
        fromfile="Original GEMM TTIR",
        tofile="Our CSE Prototype",
        lineterm=""
    )
)

print("=" * 65)
print("Our CSE Transformation")
print("=" * 65)

print("Original lines: ", len(original_ir.splitlines()))
print("Optimized lines:", len(optimized_ir.splitlines()))
print("Removed ops:    ", len(removed))

print("\n".join(diff[:100]))

Our CSE Transformation
Original lines:  180
Optimized lines: 173
Removed ops:     7
--- Original GEMM TTIR
+++ Our CSE Prototype
@@ -25,11 +25,10 @@
     %offs_n_7 = tt.make_range {end = 64 : i32, start = 0 : i32} : tensor<64xi32> loc(#loc62)
     %offs_n_8 = tt.splat %offs_n : i32 -> tensor<64xi32> loc(#loc63)
     %offs_n_9 = arith.addi %offs_n_8, %offs_n_7 : tensor<64xi32> loc(#loc63)
-    %offs_k = tt.make_range {end = 32 : i32, start = 0 : i32} : tensor<32xi32> loc(#loc64)
     %acc_10 = scf.for %k = %c0_i32 to %c16_i32 step %c1_i32 iter_args(%acc_18 = %acc) -> (tensor<32x64xf32>)  : i32 {
       %k_offsets = arith.muli %k, %c32_i32 : i32 loc(#loc66)
       %k_offsets_19 = tt.splat %k_offsets : i32 -> tensor<32xi32> loc(#loc67)
-      %k_offsets_20 = arith.addi %k_offsets_19, %offs_k : tensor<32xi32> loc(#loc67)
+      %k_offsets_20 = arith.addi %k_offsets_19, %offs_m_4 : tensor<32xi32> loc(#loc67)
       %a_ptrs = tt.expand_dims %offs_m_6 {axis = 1 : i32} : tensor<32xi32> -> tens

In [38]:
# Cell 23 — Validate Our CSE Transformation

import re
from collections import Counter

# 1. Removed SSA values must not remain in optimized IR
stale_uses = []

for item in removed:
    name = item["duplicate"]

    if re.search(
        rf"(?<![\w.]){re.escape(name)}(?![\w.])",
        optimized_ir
    ):
        stale_uses.append(name)

assert not stale_uses, f"Stale SSA references: {stale_uses}"

# 2. Compare operation counts
def count_ops(ir):
    pattern = re.compile(
        r"^\s*%[\w.]+\s*=\s*"
        r"(tt\.expand_dims|tt\.make_range)\b",
        re.MULTILINE
    )
    return Counter(pattern.findall(ir))

before = count_ops(original_ir)
after = count_ops(optimized_ir)
native = count_ops(after_cse)

print("=" * 65)
print("GEMM CSE Structural Verification")
print("=" * 65)

for op in sorted(set(before) | set(after)):
    print(
        f"{op:<20} "
        f"Original={before[op]:2d} "
        f"Prototype={after[op]:2d} "
        f"Native={native[op]:2d}"
    )

assert before - after == Counter({
    "tt.expand_dims": 6,
    "tt.make_range": 1
})

assert after == native

print("\nNo stale references to removed SSA values: PASSED")
print("Operation count matches native CSE: PASSED")
print("Structural checks: PASSED")

GEMM CSE Structural Verification
tt.expand_dims       Original=12 Prototype= 6 Native= 6
tt.make_range        Original= 3 Prototype= 2 Native= 2

No stale references to removed SSA values: PASSED
Operation count matches native CSE: PASSED
Structural checks: PASSED


### Module 2.5 — Native MLIR Pass Development

In [39]:
import platform
import subprocess

print("System:")
print(platform.platform())

subprocess.run(["cat", "/etc/os-release"])

print("\nAvailable MLIR packages:")

subprocess.run(
    ["bash", "-lc",
     "apt-cache search '^libmlir-[0-9]+-dev$|^mlir-[0-9]+-tools$'"],
    check=True
)

System:
Linux-6.6.122+-x86_64-with-glibc2.39

Available MLIR packages:


CompletedProcess(args=['bash', '-lc', "apt-cache search '^libmlir-[0-9]+-dev$|^mlir-[0-9]+-tools$'"], returncode=0)

In [40]:
import re
import subprocess

result = subprocess.run(
    ["apt-cache", "search", r"^libmlir-[0-9]+-dev$"],
    capture_output=True,
    text=True,
    check=True
)

versions = sorted(
    {
        int(v)
        for v in re.findall(
            r"libmlir-(\d+)-dev",
            result.stdout
        )
    },
    reverse=True
)

available = set(
    subprocess.run(
        ["apt-cache", "pkgnames"],
        capture_output=True,
        text=True,
        check=True
    ).stdout.splitlines()
)

compatible = [
    v for v in versions
    if all(
        f"{prefix}-{v}{suffix}" in available
        for prefix, suffix in [
            ("libmlir", "-dev"),
            ("mlir", "-tools"),
            ("llvm", "-dev"),
            ("clang", "")
        ]
    )
]

if not compatible:
    print("No complete MLIR development package set found.")
    print("We need to check Ubuntu repositories before installing.")
else:
    version = compatible[0]
    print("Selected LLVM/MLIR version:", version)

    packages = [
        f"libmlir-{version}-dev",
        f"mlir-{version}-tools",
        f"llvm-{version}-dev",
        f"clang-{version}",
        "ninja-build",
    ]

    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=True
    )

    subprocess.run(
        ["apt-get", "install", "-y", *packages],
        check=True
    )

Selected LLVM/MLIR version: 20


In [41]:
import shutil
import subprocess

print("=" * 65)
print("Native LLVM / MLIR Environment Verification")
print("=" * 65)

if "version" in globals():

    tools = [
        f"clang++-{version}",
        f"llvm-config-{version}",
        f"mlir-opt-{version}",
    ]

    for tool in tools:
        path = shutil.which(tool)
        print(f"{tool:<22}: {path or 'NOT FOUND'}")

        if path:
            subprocess.run([path, "--version"])
else:
    print("No LLVM/MLIR version selected.")

Native LLVM / MLIR Environment Verification
clang++-20            : /usr/bin/clang++-20
llvm-config-20        : /usr/bin/llvm-config-20
mlir-opt-20           : /usr/bin/mlir-opt-20


In [42]:
from pathlib import Path

Path("native_mlir").mkdir(exist_ok=True)

mlir_source = """
module {
  func.func @gemm_address(
      %row: i32,
      %col: i32,
      %stride: i32) -> i32 {

    %0 = arith.muli %row, %stride : i32
    %1 = arith.muli %row, %stride : i32

    %2 = arith.addi %0, %col : i32
    %3 = arith.addi %1, %col : i32

    %4 = arith.addi %2, %3 : i32

    return %4 : i32
  }
}
"""

Path("native_mlir/gemm_address.mlir").write_text(mlir_source)

print(mlir_source)


module {
  func.func @gemm_address(
      %row: i32,
      %col: i32,
      %stride: i32) -> i32 {

    %0 = arith.muli %row, %stride : i32
    %1 = arith.muli %row, %stride : i32

    %2 = arith.addi %0, %col : i32
    %3 = arith.addi %1, %col : i32

    %4 = arith.addi %2, %3 : i32

    return %4 : i32
  }
}



In [43]:
import subprocess

result = subprocess.run(
    [
        "mlir-opt-20",
        "native_mlir/gemm_address.mlir",
        "-o",
        "/dev/null"
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)
print(result.stderr or "Original MLIR: PASSED")

assert result.returncode == 0

Return code: 0
Original MLIR: PASSED


In [44]:
%%writefile native_mlir/GemmCSE.cpp

#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/IR/Builders.h"
#include "mlir/Pass/Pass.h"
#include "mlir/Tools/Plugins/PassPlugin.h"

#include "llvm/ADT/DenseMap.h"
#include "llvm/Support/raw_ostream.h"

using namespace mlir;

namespace {

struct GemmCSEPass
    : public PassWrapper<GemmCSEPass,
                         OperationPass<func::FuncOp>> {

    MLIR_DEFINE_EXPLICIT_INTERNAL_INLINE_TYPE_ID(GemmCSEPass)

    StringRef getArgument() const final {
        return "gemm-cse";
    }

    StringRef getDescription() const final {
        return "Eliminate duplicate GEMM integer multiplications";
    }

    void runOnOperation() override {

        func::FuncOp function = getOperation();

        int removed = 0;

        for (Block &block : function.getBlocks()) {

            // Key: pair of SSA operands
            using Key = std::pair<Value, Value>;

            llvm::DenseMap<Key, Value> seen;

            // Early-increment iteration allows safe erasure.
            for (Operation &op :
                 llvm::make_early_inc_range(block)) {

                auto mul = dyn_cast<arith::MulIOp>(&op);

                if (!mul)
                    continue;

                Key key = {
                    mul.getLhs(),
                    mul.getRhs()
                };

                auto it = seen.find(key);

                if (it == seen.end()) {
                    seen[key] = mul.getResult();
                    continue;
                }

                // Existing operation dominates the duplicate
                // because both are in the same block,
                // and we scan in instruction order.
                Value canonical = it->second;

                mul.getResult().replaceAllUsesWith(canonical);

                mul.erase();

                removed++;
            }
        }

        llvm::errs()
            << "[GemmCSE] Removed "
            << removed
            << " redundant multiplications\n";
    }
};

} // namespace

extern "C" ::mlir::PassPluginLibraryInfo
mlirGetPassPluginInfo() {

    return {
        MLIR_PLUGIN_API_VERSION,
        "GemmCSE",
        "0.1",
        []() {
            PassRegistration<GemmCSEPass>();
        }
    };
}

Overwriting native_mlir/GemmCSE.cpp


In [45]:
%%writefile native_mlir/CMakeLists.txt

cmake_minimum_required(VERSION 3.20)

project(GemmCSE LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)

find_package(LLVM 20 REQUIRED CONFIG)
find_package(MLIR 20 REQUIRED CONFIG)

add_library(GemmCSE MODULE GemmCSE.cpp)

target_include_directories(GemmCSE PRIVATE
    ${LLVM_INCLUDE_DIRS}
    ${MLIR_INCLUDE_DIRS}
)

target_compile_options(GemmCSE PRIVATE
    -fno-rtti
)

target_link_libraries(GemmCSE PRIVATE
    MLIR
)

set_target_properties(GemmCSE PROPERTIES
    PREFIX "lib"
)

Overwriting native_mlir/CMakeLists.txt


In [46]:
import subprocess
from pathlib import Path

build_dir = Path("native_mlir/build")

configure = subprocess.run(
    [
        "cmake",
        "-S", "native_mlir",
        "-B", str(build_dir),
        "-DCMAKE_BUILD_TYPE=Release",
        "-DCMAKE_CXX_COMPILER=/usr/bin/clang++-20",
        "-DLLVM_DIR=/usr/lib/llvm-20/lib/cmake/llvm",
        "-DMLIR_DIR=/usr/lib/llvm-20/lib/cmake/mlir",
    ],
    capture_output=True,
    text=True
)

print("CMake return code:", configure.returncode)
print(configure.stdout[-3000:])
print(configure.stderr[-3000:])

if configure.returncode == 0:

    build = subprocess.run(
        [
            "cmake",
            "--build",
            str(build_dir),
            "-j", "2"
        ],
        capture_output=True,
        text=True
    )

    print("\nBuild return code:", build.returncode)
    print(build.stdout[-3000:])
    print(build.stderr[-5000:])

CMake return code: 1
-- Configuring incomplete, errors occurred!

CMake Error at CMakeLists.txt:9 (find_package):
  Could not find a configuration file for package "LLVM" that is compatible
  with requested version "20".

  The following configuration files were considered but not accepted:

    /usr/lib/llvm-20/lib/cmake/llvm/LLVMConfig.cmake, version: 20.1.2
    /usr/lib/llvm-20/cmake/LLVMConfig.cmake, version: 20.1.2
    /lib/llvm-20/cmake/LLVMConfig.cmake, version: 20.1.2






In [47]:
%%writefile native_mlir/CMakeLists.txt

cmake_minimum_required(VERSION 3.20)

project(GemmCSE LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 17)
set(CMAKE_CXX_STANDARD_REQUIRED ON)

# Use the installed LLVM/MLIR toolchain.
# LLVM 20.1.2 was detected in Colab.
find_package(LLVM REQUIRED CONFIG)
find_package(MLIR REQUIRED CONFIG)

message(STATUS "LLVM version: ${LLVM_PACKAGE_VERSION}")
message(STATUS "LLVM directory: ${LLVM_DIR}")
message(STATUS "MLIR directory: ${MLIR_DIR}")

add_library(GemmCSE MODULE GemmCSE.cpp)

target_include_directories(GemmCSE PRIVATE
    ${LLVM_INCLUDE_DIRS}
    ${MLIR_INCLUDE_DIRS}
)

target_compile_options(GemmCSE PRIVATE
    -fno-rtti
)

target_link_libraries(GemmCSE PRIVATE
    MLIR
)

set_target_properties(GemmCSE PROPERTIES
    PREFIX "lib"
)

Overwriting native_mlir/CMakeLists.txt


In [48]:
import subprocess
import shutil
from pathlib import Path

build_dir = Path("native_mlir/build")

# Remove the failed CMake configuration
if build_dir.exists():
    shutil.rmtree(build_dir)

# Configure
configure = subprocess.run(
    [
        "cmake",
        "-S", "native_mlir",
        "-B", str(build_dir),
        "-DCMAKE_BUILD_TYPE=Release",
        "-DCMAKE_CXX_COMPILER=/usr/bin/clang++-20",
        "-DLLVM_DIR=/usr/lib/llvm-20/lib/cmake/llvm",
        "-DMLIR_DIR=/usr/lib/llvm-20/lib/cmake/mlir",
    ],
    capture_output=True,
    text=True
)

print("=" * 65)
print("CMake Configuration")
print("=" * 65)

print("Return code:", configure.returncode)
print(configure.stdout[-3000:])
print(configure.stderr[-3000:])

# Build only if configuration succeeded
if configure.returncode == 0:

    build = subprocess.run(
        [
            "cmake",
            "--build",
            str(build_dir),
            "-j", "2"
        ],
        capture_output=True,
        text=True
    )

    print("\n" + "=" * 65)
    print("Native MLIR Pass Build")
    print("=" * 65)

    print("Return code:", build.returncode)
    print(build.stdout[-3000:])
    print(build.stderr[-5000:])

CMake Configuration
Return code: 0
-- The CXX compiler identification is Clang 20.1.2
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/clang++-20 - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Performing Test HAVE_FFI_CALL
-- Performing Test HAVE_FFI_CALL - Success
-- Found FFI: /usr/lib/x86_64-linux-gnu/libffi.so
-- Could NOT find LibEdit (missing: LibEdit_INCLUDE_DIRS LibEdit_LIBRARIES) 
-- Found ZLIB: /usr/lib/x86_64-linux-gnu/libz.so (found version "1.3")
-- Found zstd: /usr/lib/x86_64-linux-gnu/libzstd.so
-- Found LibXml2: /usr/lib/x86_64-linux-gnu/libxml2.so (found version "2.9.14")
-- Found CURL: /usr/lib/x86_64-linux-gnu/libcurl.so (found version "8.5.0")
-- Could NOT find LibEdit (missing: LibEdit_INCLUDE_DIRS LibEdit_LIBRARIES) 
-- LLVM version: 20.1.2
-- LLVM directory: /usr/lib/llvm-20/lib/cmake/llvm
-- MLIR directory: /usr/lib/llvm-20/lib/cmake/mlir
-- Configuri

In [49]:
from pathlib import Path
import subprocess

plugin = Path(
    "native_mlir/build/libGemmCSE.so"
)

assert plugin.exists(), "GemmCSE plugin was not built."

result = subprocess.run(
    [
        "mlir-opt-20",
        f"--load-pass-plugin={plugin.resolve()}",
        "--pass-pipeline=builtin.module(func.func(gemm-cse))",
        "native_mlir/gemm_address.mlir",
        "--verify-each",
    ],
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

print("\nOptimized MLIR:\n")
print(result.stdout)

print("\nCompiler diagnostics:\n")
print(result.stderr)

Return code: 0

Optimized MLIR:

module {
  func.func @gemm_address(%arg0: i32, %arg1: i32, %arg2: i32) -> i32 {
    %0 = arith.muli %arg0, %arg2 : i32
    %1 = arith.addi %0, %arg1 : i32
    %2 = arith.addi %0, %arg1 : i32
    %3 = arith.addi %1, %2 : i32
    return %3 : i32
  }
}



Compiler diagnostics:

[GemmCSE] Removed 1 redundant multiplications



### Native C++ CSE Pass

In [50]:
%%writefile native_mlir/GemmCSE.cpp

#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Pass/Pass.h"
#include "mlir/Tools/Plugins/PassPlugin.h"

#include "llvm/ADT/DenseMap.h"
#include "llvm/ADT/STLExtras.h"
#include "llvm/Support/raw_ostream.h"

using namespace mlir;

namespace {

struct GemmCSEPass
    : public PassWrapper<
          GemmCSEPass,
          OperationPass<func::FuncOp>> {

    MLIR_DEFINE_EXPLICIT_INTERNAL_INLINE_TYPE_ID(GemmCSEPass)

    StringRef getArgument() const final {
        return "gemm-cse";
    }

    StringRef getDescription() const final {
        return "CSE for GEMM integer address arithmetic";
    }

    void runOnOperation() override {

        func::FuncOp function = getOperation();

        int removedMul = 0;
        int removedAdd = 0;
        int removedSub = 0;

        using Key = std::pair<Value, Value>;
        using Table = llvm::DenseMap<Key, Value>;

        // Process one basic block at a time.
        for (Block &block : function.getBlocks()) {

            Table mulTable;
            Table addTable;
            Table subTable;

            for (Operation &op :
                 llvm::make_early_inc_range(block)) {

                Table *table = nullptr;
                int *counter = nullptr;

                if (isa<arith::MulIOp>(op)) {
                    table = &mulTable;
                    counter = &removedMul;
                }
                else if (isa<arith::AddIOp>(op)) {
                    table = &addTable;
                    counter = &removedAdd;
                }
                else if (isa<arith::SubIOp>(op)) {
                    table = &subTable;
                    counter = &removedSub;
                }
                else {
                    continue;
                }

                // Each supported operation has two operands
                // and one result.
                Value lhs = op.getOperand(0);
                Value rhs = op.getOperand(1);

                Key key = {lhs, rhs};

                auto it = table->find(key);

                if (it == table->end()) {
                    (*table)[key] = op.getResult(0);
                    continue;
                }

                Value canonical = it->second;

                // Replace all uses of the duplicate value.
                op.getResult(0).replaceAllUsesWith(
                    canonical
                );

                // Delete the redundant operation.
                op.erase();

                (*counter)++;
            }
        }

        llvm::errs()
            << "[GemmCSE] "
            << "mul=" << removedMul
            << " add=" << removedAdd
            << " sub=" << removedSub
            << " total="
            << removedMul + removedAdd + removedSub
            << "\n";
    }
};

} // namespace

extern "C" ::mlir::PassPluginLibraryInfo
mlirGetPassPluginInfo() {

    return {
        MLIR_PLUGIN_API_VERSION,
        "GemmCSE",
        "0.2",
        []() {
            PassRegistration<GemmCSEPass>();
        }
    };
}

Overwriting native_mlir/GemmCSE.cpp


In [51]:
import subprocess
from pathlib import Path

build = subprocess.run(
    [
        "cmake",
        "--build",
        "native_mlir/build",
        "-j",
        "2"
    ],
    capture_output=True,
    text=True
)

print("Build return code:", build.returncode)
print(build.stdout[-3000:])
print(build.stderr[-5000:])

assert build.returncode == 0, "Native MLIR Pass build failed"

plugin = Path("native_mlir/build/libGemmCSE.so")

assert plugin.exists()

print("\nNative GEMM CSE Plugin: READY")

Build return code: 0
[ 50%] Building CXX object CMakeFiles/GemmCSE.dir/GemmCSE.cpp.o
[100%] Linking CXX shared module libGemmCSE.so
[100%] Built target GemmCSE



Native GEMM CSE Plugin: READY


### Native MLIR Regression Tests

In [52]:
%%writefile native_mlir/GemmCSE.cpp

#include "mlir/Dialect/Arith/IR/Arith.h"
#include "mlir/Dialect/Func/IR/FuncOps.h"
#include "mlir/Pass/Pass.h"
#include "mlir/Tools/Plugins/PassPlugin.h"

#include "llvm/ADT/DenseMap.h"
#include "llvm/ADT/STLExtras.h"
#include "llvm/Support/raw_ostream.h"

using namespace mlir;

namespace {

struct GemmCSEPass
    : public PassWrapper<
          GemmCSEPass,
          OperationPass<func::FuncOp>> {

    MLIR_DEFINE_EXPLICIT_INTERNAL_INLINE_TYPE_ID(GemmCSEPass)

    StringRef getArgument() const final {
        return "gemm-cse";
    }

    StringRef getDescription() const final {
        return "CSE for GEMM integer address arithmetic";
    }

    void runOnOperation() override {

        func::FuncOp function = getOperation();

        int removedMul = 0;
        int removedAdd = 0;
        int removedSub = 0;

        using Key = std::pair<Value, Value>;
        using Table = llvm::DenseMap<Key, Value>;

        // Process one basic block at a time.
        for (Block &block : function.getBlocks()) {

            Table mulTable;
            Table addTable;
            Table subTable;

            for (Operation &op :
                 llvm::make_early_inc_range(block)) {

                Table *table = nullptr;
                int *counter = nullptr;

                if (isa<arith::MulIOp>(op)) {
                    table = &mulTable;
                    counter = &removedMul;
                }
                else if (isa<arith::AddIOp>(op)) {
                    table = &addTable;
                    counter = &removedAdd;
                }
                else if (isa<arith::SubIOp>(op)) {
                    table = &subTable;
                    counter = &removedSub;
                }
                else {
                    continue;
                }

                // Each supported operation has two operands
                // and one result.
                Value lhs = op.getOperand(0);
                Value rhs = op.getOperand(1);

                Key key = {lhs, rhs};

                auto it = table->find(key);

                if (it == table->end()) {
                    (*table)[key] = op.getResult(0);
                    continue;
                }

                Value canonical = it->second;
                Operation *canonicalOp = canonical.getDefiningOp();

                if (!canonicalOp ||
                    canonicalOp->getAttrs() != op.getAttrs() ||
                    canonical.getType() != op.getResult(0).getType()) {
                    continue;
                }

                // Replace all uses of the duplicate value.
                op.getResult(0).replaceAllUsesWith(
                    canonical
                );

                // Delete the redundant operation.
                op.erase();

                (*counter)++;
            }
        }

        llvm::errs()
            << "[GemmCSE] "
            << "mul=" << removedMul
            << " add=" << removedAdd
            << " sub=" << removedSub
            << " total="
            << removedMul + removedAdd + removedSub
            << "\n";
    }
};

} // namespace

extern "C" ::mlir::PassPluginLibraryInfo
mlirGetPassPluginInfo() {

    return {
        MLIR_PLUGIN_API_VERSION,
        "GemmCSE",
        "0.2",
        []() {
            PassRegistration<GemmCSEPass>();
        }
    };
}

Overwriting native_mlir/GemmCSE.cpp


In [53]:
import subprocess
from pathlib import Path

build = subprocess.run(
    [
        "cmake",
        "--build",
        "native_mlir/build",
        "-j",
        "2"
    ],
    capture_output=True,
    text=True
)

print("Build return code:", build.returncode)
print(build.stdout[-3000:])
print(build.stderr[-5000:])

assert build.returncode == 0, "Native MLIR Pass build failed"

plugin = Path("native_mlir/build/libGemmCSE.so")

assert plugin.exists()

print("\nNative GEMM CSE Plugin: READY")

Build return code: 0
[ 50%] Building CXX object CMakeFiles/GemmCSE.dir/GemmCSE.cpp.o
[100%] Linking CXX shared module libGemmCSE.so
[100%] Built target GemmCSE



Native GEMM CSE Plugin: READY


In [54]:
# Cell 37 — Regression Test Definitions

from pathlib import Path

test_dir = Path("native_mlir/tests")
test_dir.mkdir(parents=True, exist_ok=True)

tests = {
    "gemm_address": """
module {
  func.func @gemm_address(
      %row: i32, %col: i32, %stride: i32) -> i32 {

    %0 = arith.muli %row, %stride : i32
    %1 = arith.muli %row, %stride : i32
    %2 = arith.addi %0, %col : i32
    %3 = arith.addi %1, %col : i32
    %4 = arith.addi %2, %3 : i32
    return %4 : i32
  }
}
""",

    "different_operands": """
module {
  func.func @different_operands(
      %a: i32, %b: i32, %c: i32) -> i32 {

    %0 = arith.muli %a, %b : i32
    %1 = arith.muli %a, %c : i32
    %2 = arith.addi %0, %1 : i32
    return %2 : i32
  }
}
""",

    "multiple_users": """
module {
  func.func @multiple_users(
      %a: i32, %b: i32) -> i32 {

    %0 = arith.muli %a, %b : i32
    %1 = arith.muli %a, %b : i32
    %2 = arith.addi %1, %0 : i32
    %3 = arith.subi %1, %0 : i32
    %4 = arith.addi %2, %3 : i32
    return %4 : i32
  }
}
""",

    "subtraction_order": """
module {
  func.func @subtraction_order(
      %a: i32, %b: i32) -> i32 {

    %0 = arith.subi %a, %b : i32
    %1 = arith.subi %b, %a : i32
    %2 = arith.addi %0, %1 : i32
    return %2 : i32
  }
}
""",

    "duplicate_subtraction": """
module {
  func.func @duplicate_subtraction(
      %a: i32, %b: i32) -> i32 {

    %0 = arith.subi %a, %b : i32
    %1 = arith.subi %a, %b : i32
    %2 = arith.addi %0, %1 : i32
    return %2 : i32
  }
}
"""
}

for name, source in tests.items():
    (test_dir / f"{name}.mlir").write_text(source)

print(f"Created {len(tests)} MLIR regression tests.")

Created 5 MLIR regression tests.


In [55]:
# Cell 38 — Native MLIR Regression Test Runner

import subprocess
import re
import pandas as pd
from pathlib import Path

plugin = Path(
    "native_mlir/build/libGemmCSE.so"
).resolve()

def operation_counts(ir):
    return {
        "mul": len(re.findall(r"=\s*arith\.muli\b", ir)),
        "add": len(re.findall(r"=\s*arith\.addi\b", ir)),
        "sub": len(re.findall(r"=\s*arith\.subi\b", ir)),
    }

expected_removed = {
    "gemm_address": (1, 1, 0),
    "different_operands": (0, 0, 0),
    "multiple_users": (1, 0, 0),
    "subtraction_order": (0, 0, 0),
    "duplicate_subtraction": (0, 0, 1),
}

test_results = []

for name in tests:

    path = test_dir / f"{name}.mlir"

    original = subprocess.run(
        [
            "mlir-opt-20",
            str(path)
        ],
        capture_output=True,
        text=True,
        check=True
    )

    optimized = subprocess.run(
        [
            "mlir-opt-20",
            f"--load-pass-plugin={plugin}",
            "--pass-pipeline=builtin.module(func.func(gemm-cse))",
            "--verify-each",
            str(path)
        ],
        capture_output=True,
        text=True,
        check=True
    )

    before = operation_counts(original.stdout)
    after = operation_counts(optimized.stdout)

    removed_counts = tuple(
        before[op] - after[op]
        for op in ["mul", "add", "sub"]
    )

    passed = removed_counts == expected_removed[name]

    test_results.append({
        "Test": name,
        "Mul Removed": removed_counts[0],
        "Add Removed": removed_counts[1],
        "Sub Removed": removed_counts[2],
        "MLIR Verified": True,
        "Status": "PASS" if passed else "FAIL"
    })

    # Save optimized IR
    (test_dir / f"{name}_optimized.mlir").write_text(
        optimized.stdout
    )

    print(
        f"{name:<25} "
        f"Removed={removed_counts} "
        f"{'PASS' if passed else 'FAIL'}"
    )

    assert passed, f"Unexpected result for {name}"

regression_df = pd.DataFrame(test_results)

print("\nAll regression tests passed.")

gemm_address              Removed=(1, 1, 0) PASS
different_operands        Removed=(0, 0, 0) PASS
multiple_users            Removed=(1, 0, 0) PASS
subtraction_order         Removed=(0, 0, 0) PASS
duplicate_subtraction     Removed=(0, 0, 1) PASS

All regression tests passed.


In [56]:
# Cell 39 — Final Optimization Results Table

from pathlib import Path
import pandas as pd

# Native CSE: use actual compiled pass output
original_native = subprocess.run(
    ["mlir-opt-20", str(test_dir / "gemm_address.mlir")],
    capture_output=True,
    text=True,
    check=True
).stdout

optimized_native = (
    test_dir / "gemm_address_optimized.mlir"
).read_text()

native_before = operation_counts(original_native)
native_after = operation_counts(optimized_native)

rows = []

for op, label in [
    ("mul", "Native CSE — arith.muli"),
    ("add", "Native CSE — arith.addi"),
    ("sub", "Native CSE — arith.subi"),
]:
    b = native_before[op]
    a = native_after[op]

    rows.append({
        "Optimization": label,
        "Before": b,
        "After": a,
        "Change": a - b,
        "Reduction": f"{(b-a)/b*100:.1f}%" if b else "N/A"
    })

total_before = sum(native_before.values())
total_after = sum(native_after.values())

rows.append({
    "Optimization": "Native CSE — Total Arithmetic",
    "Before": total_before,
    "After": total_after,
    "Change": total_after - total_before,
    "Reduction": f"{(total_before-total_after)/total_before*100:.1f}%"
})

# Real Triton GEMM TTIR CSE results
ttir_before_count = sum(count_operations(before_cse).values())
ttir_after_count = sum(count_operations(after_cse).values())

rows.append({
    "Optimization": "Triton GEMM — TTIR CSE",
    "Before": ttir_before_count,
    "After": ttir_after_count,
    "Change": ttir_after_count - ttir_before_count,
    "Reduction": f"{(ttir_before_count-ttir_after_count)/ttir_before_count*100:.1f}%"
})

results_df = pd.DataFrame(rows)

print("TABLE 1 — Compiler Optimization Results")
display(results_df)

results_df.to_csv(
    "native_mlir/optimization_results.csv",
    index=False
)

TABLE 1 — Compiler Optimization Results


,Optimization,Before,After,Change,Reduction
0,Native CSE — arith.muli,2,1,-1,50.0%
1,Native CSE — arith.addi,3,2,-1,33.3%
2,Native CSE — arith.subi,0,0,0,N/A
3,Native CSE — Total Arithmetic,5,3,-2,40.0%
4,Triton GEMM — TTIR CSE,79,72,-7,8.9%


In [57]:
# Cell 40 — Compiler Optimization Summary

summary = pd.DataFrame([
    {
        "Optimization": "Constant / Canonical Simplification",
        "Mechanism": "Simplify expressions using known values",
        "GEMM Relevance": "Simplify index and address arithmetic",
        "Verified Result": "Observed compiler pipeline; not isolated"
    },
    {
        "Optimization": "CSE — TTIR",
        "Mechanism": "Reuse equivalent SSA computations",
        "GEMM Relevance": "Reuse range and tensor shape operations",
        "Verified Result": "7 redundant operations removed"
    },
    {
        "Optimization": "LICM — TTGIR",
        "Mechanism": "Hoist loop-invariant computations",
        "GEMM Relevance": "Move address and mask computations out of K loop",
        "Verified Result": "12 operations hoisted"
    },
    {
        "Optimization": "Native C++ MLIR CSE",
        "Mechanism": "Value numbering + SSA use replacement",
        "GEMM Relevance": "Deduplicate address arithmetic",
        "Verified Result": "2 operations removed in test case"
    }
])

print("TABLE 2 — Compiler Optimization Comparison")
display(summary)

summary.to_csv(
    "native_mlir/compiler_optimization_summary.csv",
    index=False
)

TABLE 2 — Compiler Optimization Comparison


,Optimization,Mechanism,GEMM Relevance,Verified Result
0,Constant / Canonical Simplification,Simplify expressions using known values,Simplify index and address arithmetic,Observed compiler pipeline; not isolated
1,CSE — TTIR,Reuse equivalent SSA computations,Reuse range and tensor shape operations,7 redundant operations removed
2,LICM — TTGIR,Hoist loop-invariant computations,Move address and mask computations out of K loop,12 operations hoisted
3,Native C++ MLIR CSE,Value numbering + SSA use replacement,Deduplicate address arithmetic,2 operations removed in test case


### GEMM LICM Optimization Comparison

In [58]:
# Cell 41 — GEMM LICM Optimization Comparison

import re
import pandas as pd
from pathlib import Path
from collections import Counter

before_licm = Path("before_licm.mlir").read_text()
after_licm = Path("after_licm.mlir").read_text()

K = 512
BLOCK_K = 32
iterations = K // BLOCK_K


# ============================================================
# 1. Extract operations inside scf.for loop
# ============================================================

def extract_loop_operations(ir):

    lines = ir.splitlines()
    loop_ops = []

    inside_loop = False
    depth = 0

    op_pattern = re.compile(
        r"^\s*%[\w.]+\s*=\s*"
        r"((?:arith|tt|ttg)\.[\w.]+)\b"
    )

    for line in lines:

        if not inside_loop and "scf.for" in line:
            inside_loop = True
            depth = line.count("{") - line.count("}")
            continue

        if inside_loop:

            if depth > 0:
                match = op_pattern.match(line)

                if match:
                    loop_ops.append(match.group(1))

            depth += line.count("{") - line.count("}")

            if depth == 0:
                inside_loop = False

    return Counter(loop_ops)


# ============================================================
# 2. Count loop operations
# ============================================================

before_counts = extract_loop_operations(before_licm)
after_counts = extract_loop_operations(after_licm)

categories = [
    "tt.expand_dims",
    "tt.splat",
    "tt.broadcast",
    "tt.addptr",
    "arith.muli",
    "arith.cmpi"
]

rows = []

for op in categories:

    before = before_counts[op]
    after = after_counts[op]

    hoisted = before - after

    rows.append({
        "Operation": op,
        "Before (in loop)": before,
        "After (in loop)": after,
        "Hoisted": hoisted,
        "Loop iterations": iterations,
        "Est. dynamic executions avoided": (
            hoisted * (iterations - 1)
        )
    })

licm_df = pd.DataFrame(rows)

print("=" * 90)
print("TABLE 2 — GEMM LICM Optimization Results")
print("=" * 90)

display(licm_df)

print("\nTotal operations moved out of loop:",
      licm_df["Hoisted"].sum())

print(
    "Estimated repeated IR evaluations avoided:",
    licm_df["Est. dynamic executions avoided"].sum()
)

licm_df.to_csv(
    "native_mlir/licm_optimization_results.csv",
    index=False
)

TABLE 2 — GEMM LICM Optimization Results


,Operation,Before (in loop),After (in loop),Hoisted,Loop iterations,Est. dynamic executions avoided
0,tt.expand_dims,4,2,2,16,30
1,tt.splat,4,2,2,16,30
2,tt.broadcast,8,4,4,16,60
3,tt.addptr,4,3,1,16,15
4,arith.muli,3,2,1,16,15
5,arith.cmpi,4,2,2,16,30



Total operations moved out of loop: 12
Estimated repeated IR evaluations avoided: 180
